# Entraînement du Temporal Fusion Transformer

Ce notebook détaille toutes les étapes pour entraîner le modèle Temporal Fusion Transformer sur les jeux de données pickle présents dans le dossier `datasets/`.


## Configuration Git & installation rapides

Les commandes ci-dessous reproduisent exactement les étapes demandées (passage dans `/content`, clonage du dépôt, installation editable, activation d’`autoreload`). Définissez éventuellement `TRADER_AUTO_REPO`, `TRADER_AUTO_BRANCH` ou `TRADER_AUTO_CLONE_DIR` pour personnaliser l’URL, la branche ou le dossier cible avant d’exécuter la cellule suivante.


In [ ]:
import os
import sys
from pathlib import Path
from urllib.parse import urlparse

repo_url = os.environ.get(
    "TRADER_AUTO_REPO", "https://github.com/clementremillieux/trader.git"
)
repo_branch = os.environ.get("TRADER_AUTO_BRANCH", "crypto_V3")
clone_dir = os.environ.get("TRADER_AUTO_CLONE_DIR")

parsed = urlparse(repo_url)
repo_name = Path(parsed.path).name or "repo"
if repo_name.endswith(".git"):
    repo_name = repo_name[:-4]
if not clone_dir:
    clone_dir = repo_name

print("Configuration utilisée:")
print(f"  URL     : {repo_url}")
print(f"  Branche : {repo_branch}")
print(f"  Dossier : {clone_dir}")

%cd /content
!rm -rf {clone_dir}
!git clone --branch {repo_branch} {repo_url} {clone_dir}
%cd /content/{clone_dir}

%cd /content/trader

%pip install -U \
  "pandas>=2.2.3,<3" "yfinance>=0.2.56,<0.3" "scikit-learn>=1.6.1,<2" \
  "python-dotenv>=1.1,<2" "numpy==1.26.4" "alpaca-py>=0.39.4,<1" \
  "matplotlib>=3.10.1,<4" "yahooquery>=2.3.7,<3" "httpx[http2]==0.28.1" \
  "python-binance>=1.0.28,<2" "binance-connector>=3.12.0,<4" \
  "pyarrow>=20,<21" "fastparquet>=2024.11,<2025" "tqdm>=4.66.5,<5"


PROJECT_ROOT = Path("/content") / clone_dir
os.environ["TRADER_AUTO_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"TRADER_AUTO_ROOT défini sur {PROJECT_ROOT}")


In [ ]:
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from scripts.train_tft import (
    TFTConfig,
    TemporalFusionTransformer,
    compute_statistics,
    estimate_num_samples,
    list_dataset_files,
    prepare_dataloader,
    setup_logging,
    train,
)


In [ ]:
training_params = {
    "train_prefix": "full_dataset_focus_train_",
    "val_prefix": "full_dataset_focus_val_",
    "max_train_files": None,  # Ajustez pour un entrainement rapide (ex: 5)
    "max_val_files": None,
    "limit_samples_per_file": None,  # Limiter le nb de séquences par fichier si besoin
    "stat_sample_fraction": 0.2,
    "batch_size": 64,
    "epochs": 15,
    "learning_rate": 1e-4,
    "grad_clip": 1.0,
    "lambda_reg": 0.1,
    "lambda_vol": 0.02,
    "seed": 42,
    "num_workers": 2,
    "mixed_precision": torch.cuda.is_available(),
    "save_every": 0,
    "focal_gamma": 2.5,
    "focal_gamma_per_class": [3.0, 2.0, 3.0],
    "focal_alpha": [1.4, 1.0, 1.6],
    "label_smoothing": [0.05, 0.02, 0.05],
    "balance_classes": True,
    "curriculum_epochs": 3,
    "early_stopping_patience": 4,
    "hidden_dim": 384,
    "num_heads": 6,
    "num_transformer_blocks": 5,
    "dropout": 0.3,
    "static_dim": 192,
}
output_dir = PROJECT_ROOT / "models" / "tft_notebook"
output_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps(training_params, indent=2, ensure_ascii=False))
print(f"Les checkpoints seront sauvegardés dans: {output_dir}")


In [ ]:
train_files = list_dataset_files(
    path="/content/drive/MyDrive",
    prefix=training_params["train_prefix"],
    max_files=training_params["max_train_files"],
)
val_files = list_dataset_files(
    path="/content/drive/MyDrive",
    prefix=training_params["val_prefix"],
    max_files=training_params["max_val_files"],
)
print(f"Fichiers train: {len(train_files)}")
print(f"Fichiers val: {len(val_files)}")
train_files[:3], val_files[:3]


In [ ]:
from collections import Counter
import pickle


def summarize_class_distribution(file_paths, limit=None):
    counts = Counter()
    for path in file_paths:
        with open(path, "rb") as fh:
            payload = pickle.load(fh)
        labels = payload["y_cls"]
        if limit is not None:
            labels = labels[:limit]
        counts.update((labels + 1).tolist())
    total = sum(counts.values())
    return {
        cls_idx: {
            "count": counts.get(cls_idx, 0),
            "ratio": counts.get(cls_idx, 0) / total if total else 0.0,
        }
        for cls_idx in sorted(counts)
    }


train_counts = summarize_class_distribution(
    train_files, training_params["limit_samples_per_file"]
)
val_counts = summarize_class_distribution(
    val_files, training_params["limit_samples_per_file"]
)
print(
    "Distribution classes train (0=sell,1=neutre,2=buy):",
    train_counts,
)
print(
    "Distribution classes val (0=sell,1=neutre,2=buy):",
    val_counts,
)


In [ ]:
stats = compute_statistics(
    file_paths=train_files,
    sample_fraction=training_params["stat_sample_fraction"],
    seed=training_params["seed"],
    limit_samples_per_file=training_params["limit_samples_per_file"],
)
print(f"Dimension des features: {stats.feature_dim}")
print(f"Longueur de séquence: {stats.seq_len}")
print(f"Vocabulaire tau: {stats.tau_vocab_size}")
print("Poids de classes:", stats.class_weights)
print("Reg mean/std:", stats.reg_mean, stats.reg_std)
print("Vol mean/std:", stats.vol_mean, stats.vol_std)


In [ ]:
train_loader = prepare_dataloader(
    files=train_files,
    stats=stats,
    batch_size=training_params["batch_size"],
    shuffle_files=True,
    shuffle_samples=True,
    seed=training_params["seed"],
    limit_per_file=training_params["limit_samples_per_file"],
    num_workers=training_params["num_workers"],
    balance_classes=training_params["balance_classes"],
)
val_loader = prepare_dataloader(
    files=val_files,
    stats=stats,
    batch_size=training_params["batch_size"],
    shuffle_files=False,
    shuffle_samples=False,
    seed=training_params["seed"],
    limit_per_file=training_params["limit_samples_per_file"],
    num_workers=training_params["num_workers"],
    balance_classes=training_params["balance_classes"],
)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = TFTConfig(
    input_dim=stats.feature_dim,
    seq_len=stats.seq_len,
    tau_vocab_size=stats.tau_vocab_size,
    hidden_dim=training_params["hidden_dim"],
    num_heads=training_params["num_heads"],
    num_transformer_blocks=training_params["num_transformer_blocks"],
    dropout=training_params["dropout"],
    conv_kernel_sizes=(3, 5, 7),
    conv_dilations=(1, 2, 4),
    static_dim=training_params["static_dim"],
)
model = TemporalFusionTransformer(cfg).to(device)
class_weights = torch.from_numpy(stats.class_weights)
setup_logging(verbose=True)
print(model)
print(f"Device utilisé: {device}")


In [ ]:
focal_gamma_per_class = torch.tensor(
    training_params["focal_gamma_per_class"], dtype=torch.float32
)
focal_alpha = torch.tensor(training_params["focal_alpha"], dtype=torch.float32)
label_smoothing = torch.tensor(training_params["label_smoothing"], dtype=torch.float32)

best_metrics = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    class_weights=class_weights,
    device=device,
    epochs=training_params["epochs"],
    lr=training_params["learning_rate"],
    grad_clip=training_params["grad_clip"],
    lambda_reg=training_params["lambda_reg"],
    lambda_vol=training_params["lambda_vol"],
    mixed_precision=training_params["mixed_precision"],
    output_dir=output_dir,
    save_every=training_params["save_every"],
    focal_gamma=training_params["focal_gamma"],
    focal_gamma_per_class=focal_gamma_per_class,
    focal_alpha=focal_alpha,
    label_smoothing=label_smoothing,
    curriculum_epochs=training_params["curriculum_epochs"],
    early_stopping_patience=training_params["early_stopping_patience"],
)
best_metrics


In [ ]:
artifact_path = output_dir / "final_model_notebook.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": cfg.__dict__,
        "normalization": {
            "mean": stats.mean.tolist(),
            "std": stats.std.tolist(),
            "class_weights": stats.class_weights.tolist(),
            "tau_vocab_size": stats.tau_vocab_size,
            "reg_mean": stats.reg_mean,
            "reg_std": stats.reg_std,
            "vol_mean": stats.vol_mean,
            "vol_std": stats.vol_std,
            "vol_log": stats.vol_log,
        },
        "best_metrics": best_metrics,
        "params": training_params,
    },
    artifact_path,
)
print(f"Modèle sauvegardé dans: {artifact_path}")


In [ ]:
if best_metrics:
    metrics_df = pd.DataFrame(best_metrics, index=[0]).T.rename(columns={0: "valeur"})
    display(metrics_df)
    if "val_confusion" in best_metrics and best_metrics["val_confusion"]:
        confusion = np.array(best_metrics["val_confusion"])
        confusion_df = pd.DataFrame(
            confusion,
            index=["réel_-1", "réel_0", "réel_1"],
            columns=["prédit_-1", "prédit_0", "prédit_1"],
        )
        display(confusion_df)
else:
    print("Aucune métrique de validation enregistrée (best_metrics est vide).")


## Prochaines étapes

- Ajustez les hyperparamètres ou les préfixes de jeu de données pour entraîner des variantes.
- Relancez l'entraînement en changeant `max_train_files` / `max_val_files` pour un smoke test rapide.
- Analysez les checkpoints générés dans `models/tft_notebook/` ou chargez-les pour de l'inférence.
